# 03 - Model Prototyping & Baseline Tests | PM2.5 Forecasting
**Master 1 Mini Project | Academic Year 2026-2027**

Fast prototyping and initial benchmarking of baseline regression algorithms on temporal CV fold 0.

In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, Lasso, ElasticNet, HuberRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv('../data/processed/data_features.csv')
with open('../folds/folds_temporal.json', 'r') as f:
    folds = json.load(f)['folds']

fold0 = folds['fold_0']
train_idx, val_idx = fold0['train_indices'], fold0['val_indices']
num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in ['pm25', 'id', 'latitude', 'longitude']]

scaler = StandardScaler()
X_train = scaler.fit_transform(df.iloc[train_idx][num_cols].fillna(0))
y_train = df.iloc[train_idx]['pm25'].values
X_val = scaler.transform(df.iloc[val_idx][num_cols].fillna(0))
y_val = df.iloc[val_idx]['pm25'].values

models = {
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.1, random_state=42),
    'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42),
    'Huber': HuberRegressor(epsilon=1.35),
    'RandomForest': RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
}

for name, mod in models.items():
    mod.fit(X_train, y_train)
    pred = mod.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    mae = mean_absolute_error(y_val, pred)
    r2 = r2_score(y_val, pred)
    print(f'{name:15s} | RMSE: {rmse:.4f} | MAE: {mae:.4f} | R2: {r2:.4f}')
